In [ ]:
# Course setup - run once when this Colab runtime starts.
# Installs the pinned environment AND downloads the course data (via gdown -
# no Google Drive mount, no access to your Drive). Then restarts the runtime
# ONCE so the freshly installed numpy/scipy are the versions in memory
# (otherwise scipy binds against Colab's old numpy -> ufunc ABI error).
# Does nothing when run locally.
#
import os, subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    marker = Path("/content/.course_setup_done")
    if not marker.exists():
        subprocess.run(
            ["wget", "-q",
             "https://raw.githubusercontent.com/kaancet/2026KU-invivo/main/setup_colab.py"],
            check=True,
        )
        subprocess.run([sys.executable, "setup_colab.py", "--download-data"], check=True)
        marker.touch()
        print("\nSetup complete. Restarting the runtime to load the new packages.")
        print("After it restarts, run this cell again (it will skip), then continue.")
        os.kill(os.getpid(), 9)   # forces a clean Colab kernel restart
    else:
        print("Setup already done for this runtime.")


In [ ]:
# Locate the course data (downloaded into the repo in Colab; repo/data locally).
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    DATA_ROOT = Path("/content/2026KU-invivo/data")
except ModuleNotFoundError:
    DATA_ROOT = Path(sys.executable).parents[2] / "data"

DATA_PATH = DATA_ROOT / "processed"
assert DATA_PATH.exists(), f"Data not found: {DATA_PATH} (run the setup cell above first)"
print("Data path:", DATA_PATH)


In [ ]:
import os
import sys
from collections import defaultdict
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as sts
import tifffile as tf
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from piepy.imaging.onep.widefield.regions import apply_regions, load_regions, quantify, show_regions


## Load the reference image and regions of interest

In [ ]:
sessions = [
    "260818_AG001_field_coverage_test__1P_KC",
]
for s in sessions:
    ref_img = tf.imread(f"{DATA_PATH}/{s}/260615_AG001_bloodvessel_overlay_.tif")
    session_region = load_regions(f"{DATA_PATH}/{s}/{s}_ALL_AREAS.json")
    

## Load the trial-averaged movie

In [ ]:
# running through sessions
session_path = f"{DATA_PATH}/{sessions[0]}"
for fname in os.listdir(session_path):
    run_path = f"{session_path}/{fname}"
    if os.path.isdir(run_path) and run_path.endswith("_SFTF"):
        break

for path_avg in os.listdir(run_path):
    key = path_avg.removesuffix(".tif").split("_")[1]
    
    print(f"Analysing trial averaged .tif video of: {key}")
    avg_mov = tf.imread(f"{run_path}/{path_avg}")
    break


In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(avg_mov[0], cmap="viridis", animated=True)

def update(frame):
    im.set_data(avg_mov[frame])
    return [im]

ani = FuncAnimation(
    fig,
    update,
    frames=len(avg_mov),
    interval=50,   # milliseconds between frames
    blit=True
)

plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
show_regions(ref_img[:,:,0],session_region)

## Maximum activity in a given region

In [ ]:
masked = apply_regions(avg_mov, session_region)
V1_activity = masked["V1"]

In [ ]:
np.nanmin(V1_activity)

In [ ]:
fig, axs = plt.subplots(1,5,width_ratios=[4,4,1,4,1],figsize=(15,6))

show_regions(V1_activity[0],session_region,ax=axs[0])
im = axs[0].imshow(V1_activity[0], cmap="viridis", animated=True)
axs[0].set_xlim([200,400])
axs[0].set_ylim([420,230])
axs[0].set_axis_off()

def update(frame):
    im.set_data(V1_activity[frame])
    return [im]

ani = FuncAnimation(
    fig,
    update,
    frames=len(V1_activity),
    interval=50,   # milliseconds between frames
    blit=True
)


max_V1 = np.nanmax(V1_activity,axis=0)
im_max = axs[1].imshow(max_V1, cmap="viridis",vmax=np.nanmax(V1_activity),vmin=np.nanmin(V1_activity))

axs[1].set_xlim([200,400])
axs[1].set_ylim([420,230])
axs[1].set_title("Max Activity")
axs[1].set_axis_off()


std_V1 = np.nanstd(V1_activity,axis=0)
im_std = axs[3].imshow(std_V1, cmap="viridis")
axs[3].set_xlim([200,400])
axs[3].set_ylim([420,230])
axs[3].set_title("Std. Dev. Activity")
axs[3].set_axis_off()

plt.colorbar(im_max,cax=axs[2])
plt.colorbar(im_std,cax=axs[4])

plt.close(fig)
HTML(ani.to_jshtml())

##

In [ ]:
def nan_generator(shape:tuple):
    x = np.zeros(shape)
    x[:] = np.nan
    return x


POS_LOC = {"(20.0, 10.0)":0,
           "(40.0, 0.0)":1,
           "(50.0, -30.0)":2}

POS_SF = {"0.04":0,
          "0.16":1}

pos_keys = []
pos_loc = -1
mean_regions = defaultdict(partial(nan_generator,(len(sessions),  # rows: sessions
                                                  54,             # cols: frames
                                                  len(POS_LOC),   # depth: positions
                                                  len(POS_SF))))  # 4th: spatial frequency
# running through sessions
for i,s in enumerate(sessions):    
    session_path = f"{DATA_PATH}/{s}"
    print(session_path)
    for fname in os.listdir(session_path):
        run_path = f"{session_path}/{fname}"
        if os.path.isdir(run_path) and run_path.endswith("_SFTF"):
            break
    
    sesh_region = load_regions(f"{session_path}/{s}_ALL_AREAS.json")
    
    # running through positions
    for path_avg in os.listdir(run_path):
        key = path_avg.removesuffix(".tif").split("_")[1]
        key_pos = f"{key.rsplit(",",1)[0]})"
        key_sf = f"{key.rsplit(",",1)[1].strip(" ").strip(")")}"
        
        print(f"Analysing trial averaged .tif video of: {key}")
        avg_mov = tf.imread(f"{run_path}/{path_avg}")
        masked = apply_regions(avg_mov, sesh_region)
        temp = quantify(masked, 'mean', axis='spatial')
    
        for kk in temp:
            mean_regions[kk][i,:len(temp[kk]),POS_LOC[key_pos],POS_SF[key_sf]] = temp[kk]

In [ ]:
areas = ["V1","LM","AL","RL","PM","AM"]
f,axs = plt.subplots(2,len(areas),figsize=(25,10,),sharey=True)

POS_COLOR = {"(20.0, 10.0)":"#770077",
             "(40.0, 0.0)":"#0c0c0c",
             "(50.0, -30.0)":"#FF5500"}

frame_rate = 25 #Hz
frames = np.arange(0,len(mean_regions["LM"][0]))

t1 = -np.arange(0,250,40)[::-1]
t2 = np.arange(0,1900,40)
t = np.append(t1[:-1],t2)

for sf, ax_row in zip(POS_SF.keys(),axs):
    for area_name, ax_col in zip(areas,ax_row):
        area_array = mean_regions[area_name][:,:,:,POS_SF[sf]]
        for pos_name,pos_loc in POS_LOC.items():
            
            data_avg = np.nanmean(area_array[:,:,pos_loc], axis=0)
            data_sem = sts.sem(area_array[:,:,pos_loc],axis=0,nan_policy="omit")
            ax_col.plot(t,data_avg,linewidth=2,label=pos_name,color=POS_COLOR[pos_name])
            ax_col.fill_between(t,data_avg-data_sem,data_avg+data_sem,linewidth=0,color=POS_COLOR[pos_name],alpha=0.2)

        ax_col.axvline(0,color="#000000",linestyle=":")
        ax_col.set_title(area_name,fontsize=15)
        ax_col.set_xlabel("Time (ms)",fontsize=15)
        ax_col.set_ylabel(f"df/F {sf} cpd",fontsize=15)
        ax_col.tick_params(labelsize=15)
        ax_col.legend()
plt.tight_layout()

In [ ]:
auc_regions = np.load(f"{DATA_PATH}/mean_regions.npy",allow_pickle=True).item()

In [ ]:
# make an array from all the positions
x_pos = [10,20,30,40,50]
y_pos = [-30,-20,-10,0,10,20,30]
xx,yy = np.meshgrid(x_pos,y_pos[::-1])

In [ ]:
areas = ["V1","LM","AL","RL","PM","AM"]
f,axs = plt.subplots(2,len(areas),figsize=(25,10,),sharey=True,sharex=True)



for sf, ax_row in zip(POS_SF.keys(),axs):
    for area_name, ax_col in zip(areas,ax_row):
        area_array = auc_regions[area_name][:,:,:,POS_SF[sf]]
        
        data_avg = np.nanmean(area_array, axis=2)
        
        
        im = ax_col.imshow(data_avg, origin="upper",cmap="turbo",vmin=0,vmax=0.5,interpolation="sinc")
        
        ax_col.set_xticks([0,1,2,3,4])
        ax_col.set_yticks([0,1,2,3,4,5,6])
        ax_col.set_xticklabels(xx[0,:])
        ax_col.set_yticklabels(yy[:,0])
        ax_col.axhline(3,color="#000000",linestyle="-.")
        ax_col.set_title(area_name,fontsize=15)
        ax_col.set_xlabel("Azimuth (deg)",fontsize=15)
        ax_col.set_ylabel(f"{sf}cpd\nElevation (deg)",fontsize=15)
        ax_col.tick_params(labelsize=15)
        
plt.colorbar(im)
plt.tight_layout()